In [9]:
# Cell 1: Environment check + selective install (robust)
import sys
import subprocess
import os

def pip_install(spec: str):
    """Install/upgrade a package spec via pip (returns True if success)."""
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", spec])
        print(f"✅ Installed/updated: {spec}")
        return True
    except subprocess.CalledProcessError as e:
        print(f"❌ pip failed for {spec}: {e}")
        return False

def ensure_package(pip_name: str, import_name: str = None, min_version: str | None = None):
    """
    Ensure a package is installed (and >= min_version if provided).
    - pip_name: what pip installs (e.g., 'openai', 'scikit-learn')
    - import_name: what Python imports (e.g., 'openai', 'sklearn'); defaults to pip_name
    - min_version: optional minimum version string (e.g., '1.0.0')
    """
    if import_name is None:
        import_name = pip_name
    try:
        mod = __import__(import_name)
        if min_version:
            try:
                from packaging import version as _v
            except Exception:
                # packaging not present -> install then re-check
                pip_install("packaging>=23.2")
                from packaging import version as _v
            current = getattr(mod, "__version__", "0.0.0")
            if _v.parse(current) < _v.parse(min_version):
                print(f"⚠️  {import_name} {current} < required {min_version}. Upgrading…")
                pip_install(f"{pip_name}>={min_version}")
                mod = __import__(import_name)  # re-import
            else:
                print(f"✅ {import_name} {current} (meets >= {min_version})")
        else:
            ver = getattr(mod, "__version__", None)
            print(f"✅ {import_name} found" + (f" (v{ver})" if ver else ""))
        return True
    except ImportError:
        print(f"⚠️  {import_name} not found. Installing {pip_name}…")
        ok = pip_install(pip_name if not min_version else f"{pip_name}>={min_version}")
        return ok

print("🔧 Checking required packages…")

# Core data/ML stack
ensure_package("pandas", "pandas", "2.0.0")
ensure_package("numpy", "numpy", "1.24.0")
ensure_package("scikit-learn", "sklearn", "1.2.0")

# OpenAI modern client (v1+)
ensure_package("openai", "openai", "1.0.0")

# Utilities
ensure_package("python-dotenv", "dotenv", "1.0.0")
ensure_package("packaging", "packaging", "23.2")

print("\n📦 Importing libraries…")
import pandas as pd
import numpy as np
import sklearn
import openai
from dotenv import load_dotenv
from packaging import version

print("🔍 Versions:")
print(f"  pandas:        {pd.__version__}")
print(f"  numpy:         {np.__version__}")
print(f"  scikit-learn:  {sklearn.__version__}")
print(f"  openai:        {openai.__version__}")

if version.parse(openai.__version__) < version.parse("1.0.0"):
    raise RuntimeError("OpenAI package must be >= 1.0.0 for the modern API (please re-run this cell).")

print("\n🎯 Ready to proceed!")


🔧 Checking required packages…
✅ pandas 2.3.1 (meets >= 2.0.0)
✅ numpy 2.2.6 (meets >= 1.24.0)
✅ sklearn 1.7.1 (meets >= 1.2.0)
✅ openai 1.99.1 (meets >= 1.0.0)
⚠️  dotenv 0.0.0 < required 1.0.0. Upgrading…
Defaulting to user installation because normal site-packages is not writeable
✅ Installed/updated: python-dotenv>=1.0.0
✅ packaging 25.0 (meets >= 23.2)

📦 Importing libraries…
🔍 Versions:
  pandas:        2.3.1
  numpy:         2.2.6
  scikit-learn:  1.7.1
  openai:        1.99.1

🎯 Ready to proceed!


In [10]:
# =========================
# System diagnostic + label/definition setup (updated & tightened)
# =========================
import sys, platform, os, re

print("🔍 SYSTEM DIAGNOSTICS")
print("=" * 40)
print(f"Python version: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"Architecture: {platform.architecture()}")
print(f"Python executable: {sys.executable}")
print(f"CWD: {os.getcwd()}")

print("\n📦 PACKAGE VERSIONS:")
def _ver(modname, attr="__version__"):
    try:
        mod = __import__(modname)
        v = getattr(mod, attr, "unknown")
        print(f"✅ {modname}: {v}")
    except Exception as e:
        print(f"⚠️  {modname}: {e}")

_ver("pandas")
_ver("openai")
_ver("sklearn")
_ver("numpy")

# =========================
# Labels (fixed order)
# =========================
labels = [
    "Medical",
    "Mental Health",
    "Abuse",
    "Aggression",
    "Sexual",
    "Discrimination",
    "Pregnancy",
    "Not Applicable",
]

# =========================
# Tight, low-overlap definitions (optimized for small models)
# =========================
definitions = {
    "Medical": (
        "Concrete medical details tied to abortion/miscarriage/sexual-violence aftercare "
        "(e.g., pills, procedures, ultrasound/ER, antibiotics, confirmed heavy bleeding). "
        "Do not use for generic symptoms unless clearly linked to the core topics."
    ),
    "Mental Health": (
        "Explicit emotional/psychological distress caused by the core topic "
        "(e.g., panic attacks, severe anxiety, intrusive thoughts, grief)."
    ),
    "Abuse": (
        "Coercion/control/threats/sexual assault by another person. Includes pressuring about abortion, "
        "rape, stalking, intimidation, isolation, threats to report/ruin reputation. "
        "Sexual assault is Abuse (and may co-occur with Aggression)."
    ),
    "Aggression": (
        "Physical violence or explicit threats of bodily harm (hit, choke, restrain, death threats). "
        "Often co-occurs with Abuse; mark both if present."
    ),
    "Sexual": (
        "Consensual sexual content/behavior or sexual health topics (e.g., sex acts, condoms, libido). "
        "Exclude sexual assault (that belongs under Abuse/Aggression)."
    ),
    "Discrimination": (
        "Stigma, shaming, moral policing, legal/clinic barriers, or mistreatment related to the core topic "
        "(e.g., denial of care, hostile laws, derogatory remarks)."
    ),
    "Pregnancy": (
        "Being pregnant or suspected pregnant, testing, missed/late period, miscarriage/abortion status. "
        "Use when the post clearly concerns pregnancy status, risk, or loss."
    ),
    "Not Applicable": (
        "Use only if none of the above apply or the post is not about abortion, miscarriage, or sexual violence."
    ),
}

# =========================
# Domain keyword prior (used to gate relevance in prompts or QA)
# =========================
DOMAIN_KEYWORDS = {
    "abortion", "abort", "miscarriage", "pregnancy loss", "plan b",
    "mifepristone", "misoprostol", "medical abortion", "surgical abortion",
    "d&c", "dilation and curettage", "pregnant", "late period",
    "pregnancy test", "spotting", "heavy bleeding", "rape",
    "sexual assault", "assaulted", "aftercare", "clinic", "ultrasound",
}

# Optional: finer per-label evidence hints (can be sprinkled into prompts)
LABEL_HINTS = {
    "Medical": {"pill", "mifepristone", "misoprostol", "procedure", "ultrasound", "antibiotic", "infection", "ER"},
    "Mental Health": {"panic", "anxiety", "crying", "grief", "hopeless", "intrusive"},
    "Abuse": {"forced", "threat", "coerce", "stalk", "rape", "assaulted", "control", "isolate"},
    "Aggression": {"hit", "beat", "choke", "strangle", "punch", "knife", "gun", "kill"},
    "Sexual": {"sex", "condom", "libido", "porn", "kink", "after sex"},
    "Discrimination": {"illegal", "ban", "deny", "refuse", "policy", "law", "shame"},
    "Pregnancy": {"pregnant", "missed period", "late period", "test", "positive", "miscarriage"},
}

# =========================
# Helper: quick domain-relevance gate (use if you want to prefilter or add as a hint)
# =========================
DOMAIN_RE = re.compile(r"|".join(re.escape(k) for k in sorted(DOMAIN_KEYWORDS)), re.IGNORECASE)

def is_domain_related(text: str) -> bool:
    return bool(DOMAIN_RE.search(text or ""))

# =========================
# Sanity checks
# =========================
missing_defs = [l for l in labels if l not in definitions]
assert not missing_defs, f"Missing definitions for: {missing_defs}"

print("\n✅ Setup complete:")
print(f"- Labels: {len(labels)} ({labels})")
print(f"- Definitions loaded: {len(definitions)}")
print(f"- Domain keywords loaded: {len(DOMAIN_KEYWORDS)}")


🔍 SYSTEM DIAGNOSTICS
Python version: 3.10.12
Platform: Linux-6.8.0-1035-aws-x86_64-with-glibc2.35
Architecture: ('64bit', 'ELF')
Python executable: /bin/python3
CWD: /home/ubuntu

📦 PACKAGE VERSIONS:
✅ pandas: 2.3.1
✅ openai: 1.99.1
✅ sklearn: 1.7.1
✅ numpy: 2.2.6

✅ Setup complete:
- Labels: 8 (['Medical', 'Mental Health', 'Abuse', 'Aggression', 'Sexual', 'Discrimination', 'Pregnancy', 'Not Applicable'])
- Definitions loaded: 8
- Domain keywords loaded: 22


In [11]:
# =========================
# Cell 3: Load Dataset + Quick Diagnostics
# =========================
import pandas as pd

DATA_PATH = "Combined_Dataset_Annotations - Combined_Dataset.csv"

try:
    df = pd.read_csv(DATA_PATH)
    print("✅ Dataset loaded successfully!")
    print(f"📊 Shape: {df.shape[0]} rows × {df.shape[1]} columns")
    print("\n📝 Columns:")
    for col in df.columns:
        print(f" - {col}")
    print("\n🔎 Non-null counts:")
    print(df.notnull().sum())
    
    print("\n👀 Preview of data:")
    display(df.head())
    
except FileNotFoundError:
    print(f"❌ File not found: {DATA_PATH}")
    print("💡 Make sure the file is in the current directory or update the path.")
except Exception as e:
    print(f"⚠️ Error loading dataset: {e}")


✅ Dataset loaded successfully!
📊 Shape: 75 rows × 8 columns

📝 Columns:
 - id
 - soure
 - subreddit
 - title
 - body
 - created_utc
 - url
 - Tags

🔎 Non-null counts:
id             75
soure          63
subreddit      75
title          75
body           75
created_utc    75
url            75
Tags           75
dtype: int64

👀 Preview of data:


,id,soure,subreddit,title,body,created_utc,url,Tags
0,1ljxynj,abortion,abortion,Complications after abortion?,"Hi everyone, Ive read that abortions don’t cau...",2025-06-25 5:59:34,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Sexual, Pregnancy"
1,1ljxtt8,NaN,abortion,Second MA abortion today and I'm absolutely te...,I'm having my second MA abortion today and I'm...,2025-06-25 5:51:08,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Pregnancy, Mental Health"
2,1ljwhkb,abortion,abortion,Help needed/ live in Texas where abortion in b...,Anyone know of a legit site to support women i...,2025-06-25 4:31:43,https://www.reddit.com/r/abortion/comments/1lj...,"Discrimination, Pregnancy"
3,1ljvy6u,NaN,abortion,medical abortion at 6 weeks,I’ll be doing my procedure on Friday and I got...,2025-06-25 4:02:28,https://www.reddit.com/r/abortion/comments/1lj...,"Medical, Pregnancy"
4,1ljv5k4,abortion,abortion,Idk what to feel about my decision after doing...,I just had medical abortion yesterday. I was a...,2025-06-25 3:19:54,https://www.reddit.com/r/abortion/comments/1lj...,"Pregnancy, Mental Health"


In [24]:
# =========================
# Cell 4: Prompt 2 (hardened for small models)
# =========================

def make_per_label_prompt(title, body, label_name, definition):
    """
    Create a chain-of-thought prompt for evaluating a single label.

    Arguments:
    - title: Reddit post title
    - body: Reddit post description
    - label_name: The label we are testing (e.g., "Medical", "Abuse")
    - definition: The definition text for this label

    Returns:
    - Formatted prompt string for the LLM
    """

    prompt = (
    "You are an attentive content reviewer tasked with identifying specific emotional or traumatic triggers "
    "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
    "Your task is to analyze the post and decide whether readers could be triggered by it. "
    "We have multiple categories of trigger warnings, but in this case, "
    "you will evaluate **only one trigger label at a time** using a careful step-by-step approach.\n\n"
    f"Trigger Warning Label to evaluate: *{label_name}*\n\n"
    f"Definition of '{label_name}':\n"
    f"{definition.strip()}\n\n"
    "Global constraints:\n"
    "• Answer strictly with YES or NO (uppercase, no punctuation, no explanation).\n"
    "• Only mark a label YES if the post *explicitly and clearly* matches the label’s definition.\n"
    "• If the post is vague, uncertain, or does not directly match the definition, answer NO.\n"
    "• Do not infer, assume, or guess based on weak signals.\n\n"
    "Special rule for 'Not Applicable' (NA):\n"
    "• 'Not Applicable' must be marked YES **only if no other trigger labels would be YES** for this post.\n"
    "• If *any* other trigger label (evaluated separately) would be YES, then 'Not Applicable' must be NO.\n"
    "• In other words, 'Not Applicable' is mutually exclusive with all other labels.\n\n"
    "Step-by-step instructions:\n"
    "1) Carefully read the post (Title and Description).\n"
    "2) Compare the content to the label’s definition above.\n"
    "3) Decide: Does the post explicitly and clearly match this label? If yes, answer YES; otherwise, NO.\n"
    "4) If the current label is 'Not Applicable', apply the special rule above.\n\n"
    f"Title: {title.strip()}\n\n"
    f"Description: {body.strip()}\n\n"
    f"Does this post contain the '{label_name}' trigger?\n"
    "Answer:"
)

    return prompt

In [33]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import torch
import json
import pandas as pd

# ---- Globals / setup (same assumptions as your code) ----
global collected_examples_per_label
collected_examples_per_label = {}  # Reset at each call

# Assumes `labels` (list of label names) and `definitions` (dict[label] -> text) are already defined
df = pd.read_csv("annotations_dataset_with_index.csv")
embeddings = torch.load("post_embeddings.pt")
model = SentenceTransformer("all-mpnet-base-v2")
with open("similar_posts_k10.json", "r") as f:
    similar_post_map = json.load(f)


def make_multishot_prompt_cosine(k, title, body, current_index=None, include_defs=True, debug=False):
    """
    Builds one prompt per label using the EXACT template the user provided.
    Also computes top-k similar positive examples per label to populate
    `collected_examples_per_label` for external inspection, but DOES NOT
    insert examples into the prompt (to keep it exact).
    Returns: dict[label] -> prompt_str
    """
    global collected_examples_per_label
    collected_examples_per_label = {}  # reset on each call

    input_text = (title or "") + " " + (body or "")
    input_embedding = model.encode([input_text])[0]

    prompts_by_label = {}

    for label_name in labels:
        # ---- Gather positive examples for this label (for external use only) ----
        if label_name not in df.columns:
            # Keep structure predictable even if column missing
            collected_examples_per_label[label_name] = []
            if debug:
                print(f"⚠️ Skipping label '{label_name}' — column not found in dataframe.")
        else:
            trigger_examples = df[df[label_name].fillna(0) == 1]
            if current_index is not None:
                trigger_examples = trigger_examples[trigger_examples.index != current_index]

            selected_indices, selected_scores = [], []

            # Case 1: Use precomputed similar indices from JSON (if available for this post)
            used_json = False
            if current_index is not None and str(current_index) in similar_post_map:
                used_json = True
                similar_ids_with_scores = [(int(idx), float(score)) for idx, score in similar_post_map[str(current_index)]]
                for idx, score in similar_ids_with_scores:
                    if idx in trigger_examples.index:
                        selected_indices.append(idx)
                        selected_scores.append(score)
                    if len(selected_indices) == k:
                        break

            # Case 2: Compute similarity if needed
            if not selected_indices and not trigger_examples.empty:
                example_indices = trigger_examples.index.tolist()
                example_embeddings = embeddings[example_indices]
                sims = cosine_similarity([input_embedding], example_embeddings)[0]
                top_k_idx = np.argsort(sims)[-k:][::-1]
                selected_indices = [example_indices[i] for i in top_k_idx]
                selected_scores = [float(sims[i]) for i in top_k_idx]

            # Store for external consumption
            collected_examples_per_label[label_name] = []
            for idx, score in zip(selected_indices, selected_scores):
                row = df.loc[idx]
                collected_examples_per_label[label_name].append({
                    "index": int(idx),
                    "score": round(score, 4),
                    "title": row.get("title", "[No Title]"),
                    "body": row.get("body", "[No Body]")
                })
                if debug:
                    print(f"\n🔎 Example for '{label_name}' "
                          f"(source={'json' if used_json else 'cosine'}): idx={idx} score={score:.4f}")
                    print(f"• Title: {row.get('title', '[No Title]')}")
                    print(f"• Body: {row.get('body', '[No Body]')[:300]}...")
                    print("-" * 80)

        # ---- Build the EXACT prompt string for this label ----
        definition = definitions.get(label_name, "").strip()
        # Note: do not add or alter wording; keep exactly as provided
        prompt_str = (
            "You are an attentive content reviewer tasked with identifying specific emotional or traumatic triggers "
            "in Reddit posts related to abortion, miscarriage, or sexual violence.\n\n"
            "Your task is to analyze the post and decide whether readers could be triggered by it. "
            "We have multiple categories of trigger warnings, but in this case, "
            "you will evaluate **only one trigger label at a time** using a careful step-by-step approach.\n\n"
            f"Trigger Warning Label to evaluate: *{label_name}*\n\n"
            f"Definition of '{label_name}':\n"
            f"{definition.strip()}\n\n"
            "Global constraints:\n"
            "• Answer strictly with YES or NO (uppercase, no punctuation, no explanation).\n"
            "• Only mark a label YES if the post *explicitly and clearly* matches the label’s definition.\n"
            "• If the post is vague, uncertain, or does not directly match the definition, answer NO.\n"
            "• Do not infer, assume, or guess based on weak signals.\n\n"
            "Special rule for 'Not Applicable' (NA):\n"
            "• 'Not Applicable' must be marked YES **only if no other trigger labels would be YES** for this post.\n"
            "• If *any* other trigger label (evaluated separately) would be YES, then 'Not Applicable' must be NO.\n"
            "• In other words, 'Not Applicable' is mutually exclusive with all other labels.\n\n"
            "Step-by-step instructions:\n"
            "1) Carefully read the post (Title and Description).\n"
            "2) Compare the content to the label’s definition above.\n"
            "3) Decide: Does the post explicitly and clearly match this label? If yes, answer YES; otherwise, NO.\n"
            "4) If the current label is 'Not Applicable', apply the special rule above.\n\n"
            f"Title: {title.strip()}\n\n"
            f"Description: {body.strip()}\n\n"
            f"Does this post contain the '{label_name}' trigger?\n"
            "Answer:"
        )

        prompts_by_label[label_name] = prompt_str

    return prompts_by_label


# (Optional) Convenience helper if you want a single label’s prompt directly
def make_prompt_for_label(label_name, k, title, body, current_index=None, debug=False):
    pm = make_multishot_prompt_cosine(k, title, body, current_index=current_index, debug=debug)
    return pm[label_name]


In [25]:
# =========================
# Cell 5: Parsing & finalization helpers (updated for YES/NO per-label flow)
# =========================
import re

def normalize_yes_no(text: str) -> str:
    """
    Convert any LLM response into 'YES' or 'NO'.
    We expect a single-token answer, but this guards against noise.
    """
    if not text:
        return "NO"
    s = str(text).strip()
    # look for explicit YES/NO anywhere (case-insensitive)
    m = re.search(r"\b(YES|NO)\b", s, flags=re.I)
    if m:
        return m.group(1).upper()
    # fallback: first char heuristic
    return "YES" if s[:1].upper() == "Y" else "NO"


def finalize_labels(per_label_answers: dict, label_order=None):
    """
    Given a dict like {'Medical':'YES', 'Mental Health':'NO', ...},
    return the final list of labels. If none (excluding 'Not Applicable') are YES,
    return ['Not Applicable'].
    """
    if label_order is None:
        label_order = labels  # uses the global 'labels' from Cell 2

    positives = [
        lab for lab in label_order
        if per_label_answers.get(lab, "NO").strip().upper() == "YES"
        and lab != "Not Applicable"
    ]

    if positives:
        return positives
    return ["Not Applicable"]


def yesno_dict_to_binary(per_label_answers: dict, label_order=None):
    """
    Optional utility: convert {'Medical':'YES', ...} -> binary list [1/0] in label_order.
    Useful for quick metric calcs or debugging.
    """
    if label_order is None:
        label_order = labels
    return [1 if per_label_answers.get(lab, "NO").upper() == "YES" else 0 for lab in label_order]


In [26]:
# Cell 6: OpenAI Version Check and Compatibility Fix (UPDATED)

import os
import openai
from dotenv import load_dotenv
from packaging import version

# Load environment variables from .env
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

print("🔍 Checking OpenAI configuration...")

# Fail fast if the key is missing
if not OPENAI_API_KEY or not OPENAI_API_KEY.strip():
    raise RuntimeError(
        "OPENAI_API_KEY is not set. Add it to your environment or .env file before proceeding."
    )

print(f"OpenAI package version: {openai.__version__}")

# Determine API format using version
if version.parse(openai.__version__) >= version.parse("1.0.0"):
    # Modern API (v1.x)
    try:
        from openai import OpenAI
    except Exception as e:
        raise RuntimeError(f"Failed to import OpenAI client: {e}")

    client = OpenAI(api_key=OPENAI_API_KEY)
    api_format = "modern"
    print("✅ Using modern OpenAI API format (v1.x)")
else:
    # Legacy API (v0.x)
    openai.api_key = OPENAI_API_KEY
    client = openai  # keep a 'client' name for downstream code
    api_format = "legacy"
    print("✅ Using legacy OpenAI API format (v0.x)")

print(f"🔧 API Format: {api_format}")

# Optional: declare a default model name placeholder for later cells.
# Pick one your key actually has access to (you already listed available models earlier).
DEFAULT_MODEL = "gpt-4o-mini"  # change if needed per your account access

print(f"🧠 Default model set to: {DEFAULT_MODEL}")
print("🎯 OpenAI setup complete and ready to use.")


🔍 Checking OpenAI configuration...
OpenAI package version: 1.99.1
✅ Using modern OpenAI API format (v1.x)
🔧 API Format: modern
🧠 Default model set to: gpt-4o-mini
🎯 OpenAI setup complete and ready to use.


In [ ]:
# Cell 7: Robust LLM helper (uses client & DEFAULT_MODEL from Cell 6)

import time, random, re

# Fallbacks if someone runs this cell before Cell 6
try:
    client
except NameError:
    raise RuntimeError("OpenAI client not initialized. Run Cell 6 first.")

try:
    DEFAULT_MODEL
except NameError:
    DEFAULT_MODEL = "gpt-4.1-mini"   # safe default; change if your key has a different model

def _force_yes_no(text: str) -> str:
    """
    Normalize any LLM output to a strict 'YES' or 'NO'.
    Defaults to 'NO' if unclear.
    """
    if not text:
        return "NO"
    s = text.strip().upper()
    m = re.search(r"\b(YES|NO)\b", s)
    if m:
        return m.group(1)
    # simple heuristic fallback
    return "YES" if s.startswith("Y") else "NO"

def getTriggerWarningsLLMResponse(
    post_text: str,
    model: str = DEFAULT_MODEL,
    mode: str = "structured",
    temperature: float = 0.0,
    max_tokens: int = 12,       # we only need "YES"/"NO"
    verbose: bool = True,
    max_retries: int = 3,
    base_delay: float = 1.0,    # start at 1s
    max_delay: float = 20.0,
) -> str:
    """
    Calls OpenAI chat.completions with tight output control ("YES"/"NO") and
    exponential backoff (1s → 2s → 4s, capped) on rate limits.
    """
    if verbose:
        print("🧠 Classifying post for trigger warnings...")
        print("=" * 60)
    if mode == "binary":
        system_msg = "Answer ONLY with YES or NO. No punctuation, no explanation."
    else:
        system_msg = (
            "Follow the user’s instructions exactly. For EACH label block you see, write a line "
            "in the form `Answer: YES` or `Answer: NO`. At the very end, include a final line in the form "
            "`Labels: [<comma-separated list of applicable labels or Not Applicable>]`. Do not omit any labels."
        )

    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=model,
                temperature=temperature,
                max_tokens=max_tokens,
                # messages=[
                #     {
                #         "role": "system",
                #         "content": (
                #             "Answer ONLY with YES or NO. No punctuation, no explanation. "
                #             "If uncertain, answer NO."
                #         )
                #     },
                #     {"role": "user", "content": post_text},
                # ],
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": post_text},
                ],
            )
            raw = (resp.choices[0].message.content or "").strip()
            return _force_yes_no(raw)

        except Exception as e:
            # Try to respect Retry-After header if present
            retry_after = None
            try:
                retry_after = float(getattr(getattr(e, "response", None), "headers", {}).get("retry-after", 0))
            except Exception:
                pass

            msg = str(e)
            if verbose:
                print(f"⚠️ Error: {msg}")

            low = msg.lower()
            if "rate limit" in low or "429" in low:
                if retry_after and retry_after > 0:
                    wait = retry_after
                else:
                    # Exponential backoff + jitter
                    wait = min(base_delay * (2 ** attempt), max_delay)
                    wait *= (0.8 + 0.4 * random.random())
                if verbose:
                    print(f"⏳ Rate limit; retrying in {wait:.2f}s (attempt {attempt+1}/{max_retries})")
                time.sleep(wait)
                continue

            # Quota/auth/invalid model → do not loop forever
            if verbose:
                print("❌ Non-retryable error; returning 'NO'.")
            return "NO"

    if verbose:
        print("❌ Gave up after retries; returning 'NO'.")
    return "NO"


# Optional tiny throttle for batch loops to reduce 429s
def tiny_throttle(seconds: float = 0.2):
    """Call this between requests in big loops to avoid 429s."""
    time.sleep(seconds)

In [29]:
# Cell 8: Chain-of-thought prompting for each label
import time
import pandas as pd

# Sample post for evaluation
title = "Push back against Texas' use of AI to harm women"
sample_post = "Machine learning has the potential to take the world to a more utopian place, where fundamental questions about chemistry, biology, and physics are answered orders of magnitude faster than would be otherwise possible. It also has the potential to create a fundamental dystopia for all but a handful of owners who establish which rules their AIs use. Indeed, this latter scenario is already playing out globally as every “AI” badged platform being controlled on the back end to provide answers which align with their creator’s personal philosophies. This would be a matter of ethics and market forces if it were not for the real world implications of such technologies and philosophies. Texas’s anti-abortion regulations are a dystopian example, with their searching of national traffic cameras a dangerous move that means any woman in Texas has her body controlled by an inscrutable algorithm. This use of technology to control women’s bodies has been a long time coming given the anti-abortion rhetoric entrenched in US right-wing politics. Various technology companies have released period tracking apps, which while well intentioned can and have been used to keep tabs on women’s menstrual cycles with respects to miscarriages. In a society which subjugates women back to being brood mares the use of AI has made it all the more dystopian because it every application can be subtly altered to keep track of women irrespective of whether they give their explicit consent to be observedHerein lies the trap that AI is setting out for women’s pluralistic rights. It is not enough to for women to opt into wearing devices; if she is tracked and followed across an entire nation for exercising her personal autonomy it is only a few steps away from circumscribing her movement if an app considers her pregnant. It is not unthinkable that in countries such as China where AI has become embedded in transport, access to facilities, and public services that those systems can tweak what a woman can and cannot do based on the potential she may be pregnant. Given that most early pregnancies either end in miscarriage or are flushed out by the body without a woman noticing this becomes very dark indeed. You do not even need to step inside *The Handmaid’s Tale* to see the implicit and implied danger. Modern machine learning is dumb as a box of rocks when it comes to contextualising human lives. As much as we want it to be responsive to our needs ultimately most systems still operate on a normative understanding using a best fit model. No woman is the same as any other, thus what may work for 95% of women inherently excludes all other women. Texas’ restrictions on abortion make a normative assumption that women *ought* to get pregnant and keep the baby, stripping every Texan woman of her right to bodily autonomy. Under this normative framework AI is being used to surveil all women on the assumption that she could get pregnant at any moment and must be observed for her own good. When society becomes an open prison AI becomes the warder. What Texas has done with using traffic cameras and machine learning to track women who have had abortions is tantamount to creating that open prison. No Texan woman is now safe in her own body. It does not take a leap to suggest that at a state and Federal level a small change in data protection and HIPA laws and regulations would change period collection data and medical records so they can be used to enforce anti-abortion and maternity laws. Indeed, one only has to look at the Fugitive Slave Act 1850 to see how the Federal government could carve out a “settlement” between pro- and anti-abortion states, or, indeed, use Interpol red notices to extradite American women who have abortions overseas. If you think this is hyperbole look at how international immigration systems are integrating machine learning into core functionality to “protect” borders. Again, it would not take a leap to install a heat mapper in X-Ray machines to track where a woman is in her menstrual cycle based on normative assumptions, and if she fails to mean an arbitrary parameter she is then pulled from the line for an enforced pregnancy test. Machine learning can enforce normative assumptions based on what those on the backend assume to be normative human bodies. It is not beyond the realms of possibility that ICE and US border agents at some point in the future stop pregnant women from leaving the US based on anti-abortion states’ denying a woman a right to leave. In fiction most dystopias are already in media res, we rarely see in real time how they arose. Even real world examples such as the Third Reich, the Taliban, North Korea, and the People’s Republic of Iran took time and popular willingness to achieve. Texas has been heading in this direction for two decades, indeed, it is arguable that right from its independence it has been weaponizing minorities to mollify white settlers and white citizens. Now all women are on the receiving end of a creeping dystopia which machine learning and AI are the tools of choice. Keeping perspective is hard to do, especially when what is reported in public is likely the tip of a much larger issue. Unless there is scrutiny and accountability of the systems which are being set over women’s life all women face the hidden boundaries being laid out for them from the onset of puberty to menopause. At no point are men facing surveillance over vasectomies or masturbation; indeed, can you imagine the uproar if men’s health data was pulled to see if men were spilling their seed or not fulfilling their reproductive potential. You can see the X-Ray exemptions for “virile” young men, along with the prescriptions for red meat and eggs just to keep the swimmers going. All keep in rhythm by AI. Lordy. Women’s bodies have always been policed, and the hidden AI warders are insidious because women are being sold these technologies as the panacea to their period problems. Companies want women to consume their products, use them, and then willingly give their personal data which is then inscrutably used for whatever purpose the company deems necessary. Texas and other anti-abortion states want that data precisely because it will given them control, which is why many feminist organisations have recommended women ditch those apps if they live in anti-abortion states. It is not enough to state the dangers AI pose to women’s bodies. What is needed is a concerted effort to remove even the possibility of society becoming an open prison for women. No-one wants to willingly live in North Korea or Iran unless they are inculcated into the state’s ideology. Texas and other deep red states are getting that way, fetishising women’s purity and fecundity at the expense of women themselves. We cannot let AI and machine learning be weaponised against women, especially when even driving in your car becomes a dangerous exercise."
body = sample_post

# Store results
final_predictions = {}

# Loop through each label and run LLM on its prompt
for label in labels:
    print(f"\n🧠 Evaluating Label: {label}")
    definition = definitions[label]
    single_prompt = make_per_label_prompt(title, body, label, definition)
    response = getTriggerWarningsLLMResponse(single_prompt)

    # Normalize response
    answer = (response or "NO").strip().upper()
    if answer not in ["YES", "NO"]:
        answer = "NO"  # safety fallback
    
    final_predictions[label] = answer
    print(f"   → Model answered: {answer}")

# 🩵 Force fallback to "Not Applicable" if nothing was flagged
if all(value == "NO" for key, value in final_predictions.items() if key != "Not Applicable"):
    final_predictions["Not Applicable"] = "YES"

# 🔖 Final Multi-label Predictions
final_labels = [label for label, ans in final_predictions.items() if ans == "YES"]

if not final_labels:
    final_labels = ["Not Applicable"]

print("\n🔖 Final Multi-label Prediction:")
print(f"Labels: {final_labels}")



🧠 Evaluating Label: Medical
🧠 Classifying post for trigger warnings...
   → Model answered: NO

🧠 Evaluating Label: Mental Health
🧠 Classifying post for trigger warnings...
   → Model answered: YES

🧠 Evaluating Label: Abuse
🧠 Classifying post for trigger warnings...
   → Model answered: NO

🧠 Evaluating Label: Aggression
🧠 Classifying post for trigger warnings...
   → Model answered: NO

🧠 Evaluating Label: Sexual
🧠 Classifying post for trigger warnings...
   → Model answered: NO

🧠 Evaluating Label: Discrimination
🧠 Classifying post for trigger warnings...
   → Model answered: YES

🧠 Evaluating Label: Pregnancy
🧠 Classifying post for trigger warnings...
   → Model answered: YES

🧠 Evaluating Label: Not Applicable
🧠 Classifying post for trigger warnings...
   → Model answered: NO

🔖 Final Multi-label Prediction:
Labels: ['Mental Health', 'Discrimination', 'Pregnancy']


In [30]:
# Cell 9: Batch 0-shot labeling for 75 posts (title+body) -> CSV
import pandas as pd
import time, re, json

# Uses objects already defined in earlier cells:
# - labels (Cell 2)
# - definitions (Cell 2)
# - make_per_label_prompt(title, body, label, definition) (Cell 4)
# - getTriggerWarningsLLMResponse(prompt) (Cell 7)

INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"
OUTPUT_CSV = "predicted_labels_75_botty_prompt_gpt.csv"

def normalize_yes_no(text: str) -> str:
    """Parse any LLM response and return 'YES' or 'NO' (default NO)."""
    if not text:
        return "NO"
    s = str(text).strip()
    m = re.search(r"\b(YES|NO)\b", s, flags=re.I)
    if m:
        return m.group(1).upper()
    # fallback: first-char heuristic
    return "YES" if s.upper().startswith("Y") else "NO"

def predict_for_post(title: str, body: str):
    """Run per-label 0-shot prompts and return (final_labels, per_label_yes_no_dict)."""
    per_label = {}
    for label in labels:
        try:
            definition = definitions[label]
            prompt = make_per_label_prompt(title, body, label, definition)
            resp = getTriggerWarningsLLMResponse(prompt, verbose=False)
            per_label[label] = normalize_yes_no(resp)
        except Exception as e:
            # Fail closed on this label
            per_label[label] = "NO"
        # tiny pause to be gentle with rate limits (tune/remove as needed)
        time.sleep(0.05)

    # Final labels: if none YES (excluding NA), fallback to Not Applicable
    positives = [lbl for lbl, ans in per_label.items() if ans == "YES" and lbl != "Not Applicable"]
    final_labels = positives if positives else ["Not Applicable"]
    return final_labels, per_label

# --- Load dataset (expects 'title' and 'body' columns) ---
df = pd.read_csv(INPUT_CSV)
if not {'title','body'}.issubset(df.columns):
    raise KeyError("INPUT_CSV must contain 'title' and 'body' columns.")
subset = df.head(75).copy()
subset = subset[['title', 'body']].fillna('').astype(str)

predicted_lists = []
per_label_json  = []

n = len(subset)
for i, (_, row) in enumerate(subset.iterrows(), start=1):
    final_labels, per_label = predict_for_post(row['title'], row['body'])
    predicted_lists.append(final_labels)
    per_label_json.append(json.dumps(per_label, ensure_ascii=False))

    # progress every 5 posts (less spammy)
    if i % 5 == 0 or i == n:
        print(f"Processed {i}/{n} posts...")

# Save results to CSV (labels as JSON strings for easy downstream parsing)
out = subset.copy()
out['PredictedLabels'] = [json.dumps(x, ensure_ascii=False) for x in predicted_lists]
out['PerLabelYESNO']   = per_label_json
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"\n✅ Saved predictions to {OUTPUT_CSV}")


Processed 5/75 posts...
Processed 10/75 posts...
Processed 15/75 posts...
Processed 20/75 posts...
Processed 25/75 posts...
Processed 30/75 posts...
Processed 35/75 posts...
Processed 40/75 posts...
Processed 45/75 posts...
Processed 50/75 posts...
Processed 55/75 posts...
Processed 60/75 posts...
Processed 65/75 posts...
Processed 70/75 posts...
Processed 75/75 posts...

✅ Saved predictions to predicted_labels_75_botty_prompt_gpt.csv


In [31]:
# Cell: Precision/Recall/F1 per post using GT "Tags" column (Ubuntu-friendly)
import os
import ast
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# ---- paths (edit if your files live elsewhere) ----
INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"   # ground truth with a 'Tags' column
OUTPUT_CSV = "predicted_labels_75_botty_prompt_gpt.csv"                      # predictions with 'PredictedLabels' column
SAVE_AS    = "per_post_metrics_from_tags_botty_Prompt_gpt.csv"                         # will be saved in the current folder

# ---- load data ----
gt_df   = pd.read_csv(INPUT_CSV)
pred_df = pd.read_csv(OUTPUT_CSV)

# align to first 75 rows (adjust if you want more)
gt_subset   = gt_df.head(75).reset_index(drop=True)
pred_subset = pred_df.head(75).reset_index(drop=True)

# ---- helpers ----
def parse_listish(value):
    """
    Parse labels that might be stored as python-list strings or comma-separated strings.
    Returns a set of normalized labels.
    """
    if pd.isna(value):
        return set()
    s = str(value).strip()
    # try python list literal first
    try:
        maybe_list = ast.literal_eval(s)
        if isinstance(maybe_list, (list, tuple)):
            items = maybe_list
        else:
            items = [s]
    except Exception:
        # fallback: comma-separated
        items = [tok.strip() for tok in s.split(",") if tok.strip()]
    # normalize labels (consistent spacing/case)
    return set([str(x).strip() for x in items if str(x).strip()])

# ---- parse labels ----
if "Tags" not in gt_subset.columns:
    raise KeyError("Ground truth CSV must contain a 'Tags' column with human-annotated labels.")

gt_labels_list   = gt_subset["Tags"].apply(parse_listish).tolist()
pred_labels_list = pred_subset["PredictedLabels"].apply(parse_listish).tolist()

# build label universe
all_labels = sorted(list(set().union(*gt_labels_list, *pred_labels_list)))

# ---- compute metrics ----
rows = []
y_true_all, y_pred_all = [], []

for idx, (gt_set, pred_set) in enumerate(zip(gt_labels_list, pred_labels_list), start=1):
    y_true = [1 if l in gt_set else 0 for l in all_labels]
    y_pred = [1 if l in pred_set else 0 for l in all_labels]

    y_true_all.extend(y_true)
    y_pred_all.extend(y_pred)

    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)

    rows.append({
        "PostIndex": idx,
        "TrueLabels": sorted(list(gt_set)),
        "PredictedLabels": sorted(list(pred_set)),
        "Precision": round(p, 3),
        "Recall": round(r, 3),
        "F1": round(f, 3),
    })

metrics_df = pd.DataFrame(rows)

# macro averages = mean of per-post metrics
macro_p = float(metrics_df["Precision"].mean())
macro_r = float(metrics_df["Recall"].mean())
macro_f = float(metrics_df["F1"].mean())

# micro averages = computed on pooled one-hot vectors
micro_p = precision_score(y_true_all, y_pred_all, zero_division=0)
micro_r = recall_score(y_true_all, y_pred_all, zero_division=0)
micro_f = f1_score(y_true_all, y_pred_all, zero_division=0)

summary_rows = pd.DataFrame([
    {"PostIndex": "Macro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(macro_p, 3), "Recall": round(macro_r, 3), "F1": round(macro_f, 3)},
    {"PostIndex": "Micro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(micro_p, 3), "Recall": round(micro_r, 3), "F1": round(micro_f, 3)},
])

final_metrics = pd.concat([metrics_df, summary_rows], ignore_index=True)

# ---- save (no /mnt/data, so always local) ----
final_metrics.to_csv(SAVE_AS, index=False, encoding="utf-8")
print(f"✅ Saved per-post metrics + macro/micro averages to: {os.path.abspath(SAVE_AS)}")


✅ Saved per-post metrics + macro/micro averages to: /home/ubuntu/per_post_metrics_from_tags_botty_Prompt_gpt.csv


In [32]:
# Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_botty_Prompt_gpt.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.857
Macro Recall: 0.830
Macro F1: 0.817

Micro Precision: 0.828
Micro Recall: 0.824
Micro F1: 0.826

Perfect Matches: 28 out of 75 posts (37.3%) exactly matched human annotations
Completely Incorrect: 2 out of 75 posts (2.7%) had no correct labels


In [41]:
# Cell 9: Batch 0-shot labeling for 75 posts (title+body) -> CSV
import pandas as pd
import time, re, json

# Uses objects already defined in earlier cells:
# - labels (Cell 2)
# - definitions (Cell 2)
# - make_per_label_prompt(title, body, label, definition) (Cell 4)
# - getTriggerWarningsLLMResponse(prompt) (Cell 7)

INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"
OUTPUT_CSV = "predicted_labels_75_rosie_prompt_1_gpt.csv"

def normalize_yes_no(text: str) -> str:
    """Parse any LLM response and return 'YES' or 'NO' (default NO)."""
    if not text:
        return "NO"
    s = str(text).strip()
    m = re.search(r"\b(YES|NO)\b", s, flags=re.I)
    if m:
        return m.group(1).upper()
    # fallback: first-char heuristic
    return "YES" if s.upper().startswith("Y") else "NO"

def predict_for_post(title: str, body: str):
    """Run per-label 0-shot prompts and return (final_labels, per_label_yes_no_dict)."""
    per_label = {}
    for label in labels:
        try:
            definition = definitions[label]
            prompt = make_per_label_prompt(title, body, label, definition)
            resp = getTriggerWarningsLLMResponse(prompt, verbose=False)
            per_label[label] = normalize_yes_no(resp)
        except Exception as e:
            # Fail closed on this label
            per_label[label] = "NO"
        # tiny pause to be gentle with rate limits (tune/remove as needed)
        time.sleep(0.05)

    # Final labels: if none YES (excluding NA), fallback to Not Applicable
    positives = [lbl for lbl, ans in per_label.items() if ans == "YES" and lbl != "Not Applicable"]
    final_labels = positives if positives else ["Not Applicable"]
    return final_labels, per_label

# --- Load dataset (expects 'title' and 'body' columns) ---
df = pd.read_csv(INPUT_CSV)
if not {'title','body'}.issubset(df.columns):
    raise KeyError("INPUT_CSV must contain 'title' and 'body' columns.")
subset = df.head(75).copy()
subset = subset[['title', 'body']].fillna('').astype(str)

predicted_lists = []
per_label_json  = []

n = len(subset)
for i, (_, row) in enumerate(subset.iterrows(), start=1):
    final_labels, per_label = predict_for_post(row['title'], row['body'])
    predicted_lists.append(final_labels)
    per_label_json.append(json.dumps(per_label, ensure_ascii=False))

    # progress every 5 posts (less spammy)
    if i % 5 == 0 or i == n:
        print(f"Processed {i}/{n} posts...")

# Save results to CSV (labels as JSON strings for easy downstream parsing)
out = subset.copy()
out['PredictedLabels'] = [json.dumps(x, ensure_ascii=False) for x in predicted_lists]
out['PerLabelYESNO']   = per_label_json
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')

print(f"\n✅ Saved predictions to {OUTPUT_CSV}")


Processed 5/75 posts...
Processed 10/75 posts...
Processed 15/75 posts...
Processed 20/75 posts...
Processed 25/75 posts...
Processed 30/75 posts...
Processed 35/75 posts...
Processed 40/75 posts...
Processed 45/75 posts...
Processed 50/75 posts...
Processed 55/75 posts...
Processed 60/75 posts...
Processed 65/75 posts...
Processed 70/75 posts...
Processed 75/75 posts...

✅ Saved predictions to predicted_labels_75_rosie_prompt_1_gpt.csv


In [44]:
# Cell: Precision/Recall/F1 per post using GT "Tags" column (Ubuntu-friendly)
import os
import ast
import pandas as pd
from sklearn.metrics import precision_score, recall_score, f1_score

# ---- paths (edit if your files live elsewhere) ----
INPUT_CSV  = "Combined_Dataset_Annotations - Combined_Dataset.csv"   # ground truth with a 'Tags' column
OUTPUT_CSV = "predicted_labels_75_rosie_prompt_1_gpt.csv"                      # predictions with 'PredictedLabels' column
SAVE_AS    = "per_post_metrics_from_tags_rosie_Prompt_1_gpt.csv"                         # will be saved in the current folder

# ---- load data ----
gt_df   = pd.read_csv(INPUT_CSV)
pred_df = pd.read_csv(OUTPUT_CSV)

# align to first 75 rows (adjust if you want more)
gt_subset   = gt_df.head(75).reset_index(drop=True)
pred_subset = pred_df.head(75).reset_index(drop=True)

# ---- helpers ----
def parse_listish(value):
    """
    Parse labels that might be stored as python-list strings or comma-separated strings.
    Returns a set of normalized labels.
    """
    if pd.isna(value):
        return set()
    s = str(value).strip()
    # try python list literal first
    try:
        maybe_list = ast.literal_eval(s)
        if isinstance(maybe_list, (list, tuple)):
            items = maybe_list
        else:
            items = [s]
    except Exception:
        # fallback: comma-separated
        items = [tok.strip() for tok in s.split(",") if tok.strip()]
    # normalize labels (consistent spacing/case)
    return set([str(x).strip() for x in items if str(x).strip()])

# ---- parse labels ----
if "Tags" not in gt_subset.columns:
    raise KeyError("Ground truth CSV must contain a 'Tags' column with human-annotated labels.")

gt_labels_list   = gt_subset["Tags"].apply(parse_listish).tolist()
pred_labels_list = pred_subset["PredictedLabels"].apply(parse_listish).tolist()

# build label universe
all_labels = sorted(list(set().union(*gt_labels_list, *pred_labels_list)))

# ---- compute metrics ----
rows = []
y_true_all, y_pred_all = [], []

for idx, (gt_set, pred_set) in enumerate(zip(gt_labels_list, pred_labels_list), start=1):
    y_true = [1 if l in gt_set else 0 for l in all_labels]
    y_pred = [1 if l in pred_set else 0 for l in all_labels]

    y_true_all.extend(y_true)
    y_pred_all.extend(y_pred)

    p = precision_score(y_true, y_pred, zero_division=0)
    r = recall_score(y_true, y_pred, zero_division=0)
    f = f1_score(y_true, y_pred, zero_division=0)

    rows.append({
        "PostIndex": idx,
        "TrueLabels": sorted(list(gt_set)),
        "PredictedLabels": sorted(list(pred_set)),
        "Precision": round(p, 3),
        "Recall": round(r, 3),
        "F1": round(f, 3),
    })

metrics_df = pd.DataFrame(rows)

# macro averages = mean of per-post metrics
macro_p = float(metrics_df["Precision"].mean())
macro_r = float(metrics_df["Recall"].mean())
macro_f = float(metrics_df["F1"].mean())

# micro averages = computed on pooled one-hot vectors
micro_p = precision_score(y_true_all, y_pred_all, zero_division=0)
micro_r = recall_score(y_true_all, y_pred_all, zero_division=0)
micro_f = f1_score(y_true_all, y_pred_all, zero_division=0)

summary_rows = pd.DataFrame([
    {"PostIndex": "Macro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(macro_p, 3), "Recall": round(macro_r, 3), "F1": round(macro_f, 3)},
    {"PostIndex": "Micro-Avg", "TrueLabels": None, "PredictedLabels": None,
     "Precision": round(micro_p, 3), "Recall": round(micro_r, 3), "F1": round(micro_f, 3)},
])

final_metrics = pd.concat([metrics_df, summary_rows], ignore_index=True)

# ---- save (no /mnt/data, so always local) ----
final_metrics.to_csv(SAVE_AS, index=False, encoding="utf-8")
print(f"✅ Saved per-post metrics + macro/micro averages to: {os.path.abspath(SAVE_AS)}")


✅ Saved per-post metrics + macro/micro averages to: /home/ubuntu/per_post_metrics_from_tags_rosie_Prompt_1_gpt.csv


In [8]:
# Notebook cell: Summarize per-post metrics CSV (advisor-friendly printout)
import pandas as pd
import ast

# === Configure this to your file ===
CSV_PATH = "per_post_metrics_from_tags_rosie_Prompt_1_gpt.csv"

def _safe_list(x):
    """Parse list-like strings into Python lists; return [] if parsing fails."""
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    s = str(x).strip()
    # try JSON-ish or Python literal list
    try:
        val = ast.literal_eval(s)
        if isinstance(val, list):
            return val
    except Exception:
        pass
    # fallback: comma-split
    return [t.strip() for t in s.split(",") if t.strip()]

def summarize_metrics(csv_path: str):
    df = pd.read_csv(csv_path)

    # Detect summary rows if present
    has_macro = (df["PostIndex"] == "Macro-Avg").any() if "PostIndex" in df.columns else False
    has_micro = (df["PostIndex"] == "Micro-Avg").any() if "PostIndex" in df.columns else False

    # Pull macro/micro if available, else compute from per-post rows
    macro_p = macro_r = macro_f = None
    micro_p = micro_r = micro_f = None

    if has_macro:
        row = df.loc[df["PostIndex"] == "Macro-Avg"].iloc[0]
        macro_p, macro_r, macro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])
    if has_micro:
        row = df.loc[df["PostIndex"] == "Micro-Avg"].iloc[0]
        micro_p, micro_r, micro_f = float(row["Precision"]), float(row["Recall"]), float(row["F1"])

    # Filter per-post rows
    per_post = df.copy()
    if "PostIndex" in per_post.columns:
        per_post = per_post[~per_post["PostIndex"].isin(["Macro-Avg", "Micro-Avg"])]

    # Compute perfect/incorrect if we have labels
    perfect_matches = completely_incorrect = None
    if {"TrueLabels", "PredictedLabels"}.issubset(per_post.columns):
        t_labels = per_post["TrueLabels"].apply(_safe_list)
        p_labels = per_post["PredictedLabels"].apply(_safe_list)

        perfect_matches = sum(set(t) == set(p) for t, p in zip(t_labels, p_labels))
        completely_incorrect = sum(len(set(t).intersection(set(p))) == 0 for t, p in zip(t_labels, p_labels))
        n_posts = len(per_post)
    else:
        n_posts = len(per_post)

    # If macro/micro not present, approximate macro as mean of per-post, and skip micro
    if macro_p is None and {"Precision","Recall","F1"}.issubset(per_post.columns):
        macro_p = per_post["Precision"].mean()
        macro_r = per_post["Recall"].mean()
        macro_f = per_post["F1"].mean()

    # Pretty print in your requested format
    print("Here’s the summary you can share with your advisor:\n")
    if macro_p is not None:
        print(f"Macro Precision: {macro_p:.3f}")
        print(f"Macro Recall: {macro_r:.3f}")
        print(f"Macro F1: {macro_f:.3f}\n")
    else:
        print("Macro metrics: not available in this file.\n")

    if micro_p is not None:
        print(f"Micro Precision: {micro_p:.3f}")
        print(f"Micro Recall: {micro_r:.3f}")
        print(f"Micro F1: {micro_f:.3f}\n")
    else:
        print("Micro metrics: not available in this file.\n")

    if perfect_matches is not None:
        print(f"Perfect Matches: {perfect_matches} out of {n_posts} posts ({100*perfect_matches/n_posts:.1f}%) exactly matched human annotations")
    else:
        print("Perfect Matches: (labels not found in CSV to compute)")
    if completely_incorrect is not None:
        print(f"Completely Incorrect: {completely_incorrect} out of {n_posts} posts ({100*completely_incorrect/n_posts:.1f}%) had no correct labels")
    else:
        print("Completely Incorrect: (labels not found in CSV to compute)")

# Run
summarize_metrics(CSV_PATH)


Here’s the summary you can share with your advisor:

Macro Precision: 0.788
Macro Recall: 0.880
Macro F1: 0.808

Micro Precision: 0.753
Micro Recall: 0.882
Micro F1: 0.813

Perfect Matches: 27 out of 75 posts (36.0%) exactly matched human annotations
Completely Incorrect: 2 out of 75 posts (2.7%) had no correct labels


In [34]:
# Cell: Batch 1-shot (k=5) few-shot labeling for 75 posts -> CSV + used examples JSON
import pandas as pd
import json, time, copy, ast, re
import numpy as np

INPUT_CSV    = "Combined_Dataset_Annotations - Combined_Dataset.csv"
OUTPUT_CSV   = "predicted_labels_75_few_shot_k_5.csv"
EXAMPLES_JSON = "used_examples_for_75_posts_K_5.json"

# --- robust parsers / helpers ---
YESNO_RE = re.compile(r"\b(YES|NO)\b", re.IGNORECASE)

def extract_yes_no(resp) -> str:
    """
    Robustly coerce model output to YES/NO.
    - Accept dicts with .get('text'), .get('content'), .get('choices'[0]['text'])
    - Take only the first line, then first YES/NO token anywhere on that line.
    """
    if resp is None:
        return "NO"
    # unwrap common response shapes
    if isinstance(resp, dict):
        if "text" in resp and isinstance(resp["text"], str):
            resp = resp["text"]
        elif "content" in resp and isinstance(resp["content"], str):
            resp = resp["content"]
        elif "choices" in resp and isinstance(resp["choices"], list) and resp["choices"]:
            ch = resp["choices"][0]
            if isinstance(ch, dict):
                if "text" in ch:
                    resp = ch["text"]
                elif "message" in ch and isinstance(ch["message"], dict) and "content" in ch["message"]:
                    resp = ch["message"]["content"]
                else:
                    resp = str(ch)
            else:
                resp = str(ch)
        else:
            resp = str(resp)
    # must be string now
    if not isinstance(resp, str):
        resp = str(resp)

    line = resp.strip().splitlines()[0] if resp.strip() else ""
    m = YESNO_RE.search(line)
    if m:
        return "YES" if m.group(1).upper() == "YES" else "NO"

    # fallback: first token check
    tok = line.upper().split()[:1]
    return "YES" if (tok and tok[0] == "YES") else "NO"

def convert_to_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    return obj

def tiny_throttle(sec=0.25):
    try:
        time.sleep(sec)
    except Exception:
        pass

# --- load input posts ---
df_in = pd.read_csv(INPUT_CSV)
if not {'title','body'}.issubset(df_in.columns):
    raise KeyError("INPUT_CSV must contain 'title' and 'body' columns.")

subset = df_in.head(75).copy()
subset = subset[['title','body']].fillna('').astype(str)

predicted_lists = []
all_used_examples = []   # snapshot of examples used per post

for i, (_, row) in enumerate(subset.iterrows(), start=1):
    title = row['title']
    body  = row['body']

    # Warm up cosine neighbors (populates collected_examples_per_label)
    _ = make_multishot_prompt_cosine(
        k=5,
        title=title,
        body=body,
        current_index=None,
        include_defs=True,
        debug=False
    )

    positive_labels = []

    for j, lbl in enumerate(labels):
        single_prompt = make_prompt_for_label(
            label_name=lbl,
            k=5,
            title=title,
            body=body,
            current_index=None,
            debug=False
        )

        # First attempt (tight)
        resp = getTriggerWarningsLLMResponse(
            single_prompt,
            mode="raw",
            max_tokens=4,
            temperature=0.0,
            verbose=False
        )
        ans = extract_yes_no(resp)

        # If we didn't get a clean YES/NO on first line, retry with a slightly bigger budget
        if ans not in ("YES", "NO"):
            resp = getTriggerWarningsLLMResponse(
                single_prompt,
                mode="raw",
                max_tokens=8,   # allow newline + token quirks
                temperature=0.0,
                verbose=False
            )
            ans = extract_yes_no(resp)

        # Light logging for first 2 posts × first 3 labels to diagnose
        if i <= 2 and j < 3:
            print("\n--- DEBUG RAW ---")
            print(f"Post {i}, Label '{lbl}':")
            # Show a compact view
            if isinstance(resp, str):
                print(resp[:200].replace("\n", "\\n"))
            else:
                s = json.dumps(resp, default=str) if not isinstance(resp, str) else resp
                print(s[:200].replace("\n", "\\n"))
            print(f"Parsed: {ans}")
            print("-----------------")

        if ans == "YES":
            positive_labels.append(lbl)

        tiny_throttle(0.15)

    if not positive_labels:
        positive_labels = ["Not Applicable"]

    predicted_lists.append(positive_labels)

    snapshot = copy.deepcopy(collected_examples_per_label) if 'collected_examples_per_label' in globals() else {}
    all_used_examples.append({
        "post_index_1_based": i,
        "title": title,
        "body": body,
        "examples_by_label": snapshot
    })

    print(f"{i}. Labels: {positive_labels}")

# --- save predictions CSV ---
out = subset.copy()
out['PredictedLabels'] = [json.dumps(x, ensure_ascii=False) for x in predicted_lists]
out.to_csv(OUTPUT_CSV, index=False, encoding='utf-8')
print(f"\n✅ Saved predictions to {OUTPUT_CSV}")

# --- save used examples JSON ---
with open(EXAMPLES_JSON, "w", encoding="utf-8") as f:
    json.dump(all_used_examples, f, indent=2, ensure_ascii=False, default=convert_to_serializable)
print(f"✅ Saved used examples to {EXAMPLES_JSON}")


TypeError: getTriggerWarningsLLMResponse() got an unexpected keyword argument 'mode'